In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt

In [32]:
with open('./security_data.json') as file:
    json_data = json.load(file)

events = pd.DataFrame(json_data['events'])
entities = pd.DataFrame(json_data['entities'])
relationships = pd.DataFrame(json_data['relationships'])

attack_scenarios = pd.DataFrame(json_data['ground_truth']['attack_scenarios'])

In [33]:
events

,id,type,timestamp,severity,description,alert_style,alert_message
0,224a7b52-a126-44bc-90d4-301ab4de4664,Authentication,2025-05-30T21:55:58.061710,6,Service account used interactively,raw_telemetry,Windows Event ID 4624: Service account used in...
1,c5d5885f-5c1b-41c2-a3c4-2fa7fe4074a0,Defense Evasion,2025-05-31T05:15:42.002908,8,Logs cleared or deleted,behavioral,POTENTIAL DEFENSE EVASION: Logs cleared or del...
2,c7e0c496-a7b1-433b-8cc6-a5e4e7ad8cfa,Authentication,2025-05-31T09:40:32.084221,8,Lateral movement to critical system,raw_telemetry,RAW_TELEMETRY ALERT: Lateral movement to criti...
3,30d15fa0-0a9b-4888-85a5-f5e19950b5fa,Privilege Operation,2025-05-31T10:17:30.872790,7,Scheduled task created for persistence,raw_telemetry,Process chrome.exe started with elevated privi...
4,312b0a4c-e101-4533-9105-dd57877e7406,Data Access,2025-05-31T10:40:12.918836,9,Access to credentials store,rules_based,RULES_BASED ALERT: Access to credentials store...
...,...,...,...,...,...,...,...
1007,23bfc397-f876-4317-95f0-d54401399faa,Authentication,2025-05-25T12:44:29.058975,1,Service account used interactively,raw_telemetry,Windows Event ID 4624: Service account used in...
1008,7189d907-9157-45ed-97ae-88045deba4ed,Authentication,2025-05-26T04:47:52.549541,1,Service account used interactively,raw_telemetry,RAW_TELEMETRY ALERT: Service account used inte...
1009,7ed85397-8dcb-4108-bffd-5032799d4c3b,Authentication,2025-05-27T23:22:05.782057,3,Login outside business hours,behavioral,USER BEHAVIOR CHANGE: Login outside business h...
1010,4c1a0297-044f-4e6b-9e8b-262d68710be6,Exfiltration,2025-05-28T05:45:19.167575,1,Data compressed prior to exfiltration,raw_telemetry,RAW_TELEMETRY ALERT: Data compressed prior to ...


In [4]:
entities

,id,type,name,properties
0,fa771ab4-f8e7-46f4-8d72-d80a7a5e4996,Host,WS-878,"{'hostname': 'WS-878', 'ip_address': '10.128.1..."
1,7ef91f18-f3ab-435a-8b07-2a7ae3ca18e2,Host,DC-601,"{'hostname': 'DC-601', 'ip_address': '10.215.1..."
2,97f6f080-8eff-4514-ac14-9cd1ea3bb83b,User,Grace Brown,"{'username': 'gbrown', 'role': 'Analyst', 'dep..."
3,8e8bc3f9-97b7-433a-9461-61737109fece,File,4m8t1qoj.ps1,"{'path': 'C:\Windows\Temp', 'name': '4m8t1qoj...."
4,5d0944ac-dec0-4557-abdd-ae3bf4dbff3d,File,9tq6v9mh.pdf,"{'path': '/tmp', 'name': '9tq6v9mh.pdf', 'size..."
...,...,...,...,...
1553,49f071c5-489f-4023-a50c-c07a414393f0,File,s4q3ofyx.ps1,"{'path': 'C:\Users\Public', 'name': 's4q3ofyx...."
1554,d3810eea-f37c-4ab4-bda7-4ebc657aa53a,Domain,jbtdcmd.net,"{'name': 'jbtdcmd.net', 'registration_date': '..."
1555,89e74d25-ec84-4c6d-8669-954fb5ce38a3,NetworkConnection,10.3.252.166:443 -> 10.16.126.203,"{'source_ip': '10.3.252.166', 'dest_ip': '10.1..."
1556,ac9f47fd-0176-4b1c-abdf-beec20a31d26,Process,powershell.exe,"{'name': 'powershell.exe', 'pid': 25626, 'comm..."


In [5]:
relationships

,source,target,weight,type
0,49910d93-6d21-40b2-aa7b-abe591421e69,97f6f080-8eff-4514-ac14-9cd1ea3bb83b,1.0,related_to
1,49910d93-6d21-40b2-aa7b-abe591421e69,7ef91f18-f3ab-435a-8b07-2a7ae3ca18e2,1.0,related_to
2,49910d93-6d21-40b2-aa7b-abe591421e69,5d0944ac-dec0-4557-abdd-ae3bf4dbff3d,1.0,related_to
3,07b95b57-22c8-43f8-8a9c-de241e15cc74,8e8bc3f9-97b7-433a-9461-61737109fece,1.0,related_to
4,07b95b57-22c8-43f8-8a9c-de241e15cc74,fa771ab4-f8e7-46f4-8d72-d80a7a5e4996,1.0,related_to
...,...,...,...,...
2012,9ea4db61-24db-4c21-ab53-e3d1e8d78af5,28787680-6886-428a-aa71-bfaf94b4cc1e,1.0,related_to
2013,9ea4db61-24db-4c21-ab53-e3d1e8d78af5,d3810eea-f37c-4ab4-bda7-4ebc657aa53a,1.0,related_to
2014,9ea4db61-24db-4c21-ab53-e3d1e8d78af5,596d35aa-d718-4054-9980-bc8c8ca8948e,1.0,related_to
2015,9ea4db61-24db-4c21-ab53-e3d1e8d78af5,05b60c63-1cc0-4961-9f16-5d2f16f2238b,1.0,related_to


In [6]:
attack_scenarios

,id,description,event_ids,entity_ids
0,5f73c828-8405-4c3c-a7aa-3b6adb3596a1,Attack Scenario: Ransomware,"[e2093370-4480-4cf2-906e-68bdb37fcbf0, 2103cc2...","[aa4f416e-e992-4c65-8e74-f0c6fbe77865, 1f8283b..."
1,f65bc61d-b5fb-4f20-ad29-7ca1469c1134,Attack Scenario: Supply Chain Attack,"[62e31a95-1a6e-42a9-b028-17256c54bbea, 4a6e8b2...","[c34f4056-50cf-44b2-8334-aa11452ffcc5, 87b4676..."
2,13505ccf-edab-4ab9-a02d-2e015c931a5d,Attack Scenario: Insider Threat,"[4d734d32-7239-4ce2-b25c-96cac9bb160e, 40ba766...","[13761d1c-648a-4eae-86b1-d962a21114b0, 889ccf6..."
3,b2665855-7f04-4b9a-b2da-11ebf106d0ed,Attack Scenario: Credential Stuffing,"[ef514d71-32e5-4529-be60-ab1fe7819725, 68b35de...","[ce95d8aa-eccd-4287-bf24-1933cfb5f102, 121a7e0..."
4,d3413b88-f28b-4f02-b693-5b30f4778989,Attack Scenario: Lateral Movement Campaign,"[460e9d85-0382-4063-938e-80141586fcdd, 3977130...","[d8996ca9-91fd-452c-a547-a445ff3d17c1, c39a73a..."
5,1eee42df-b4b7-4830-b058-3761cff033ed,Attack Scenario: Data Exfiltration,"[d51bc915-f743-4020-b8a7-c86ed19718d0, 36f56ac...","[c512ae50-d5e5-4509-9386-6da189754a84, 7e8767f..."


In [34]:
'''
Sanity Check
====================================================================================
The event_ids that appear in the ground_truth.attack_scenarios array are precisely
those that are listed in ground_truth.true_positive_events.
'''

e = [event for i in range(len(attack_scenarios)) for event in attack_scenarios['event_ids'][i]]
print(set(e) == set(json_data['ground_truth']['true_positive_events'])) # should print true

'''
Likewise, the entity_ids that appear in the ground_truth.attack_scenarios array are precisely
those that are listed in ground_truth.true_positive_entities.
'''

e = [entity for i in range(len(attack_scenarios)) for entity in attack_scenarios['entity_ids'][i]]
print(set(e) == set(json_data['ground_truth']['true_positive_entities'])) # should print true

True
True


In [7]:
def create_bipartite_graph(part1, part2, edges) -> nx.Graph:
    """Load json file and return a bipartite weighted graph.

    Nodes are the `id` values from `events` and `entities`.
    Edges are created from `relationships` with the edge attribute `weight` set
    from the relationship's `weight` field. Each node gets attributes:
    - `kind`: either 'event' or 'entity'
    - `bipartite`: 0 for events, 1 for entities
    """
    G = nx.Graph()

    part1_ids = part1["id"]
    part2_ids = part2["id"]

    for nid in part1_ids:
        G.add_node(nid, bipartite=0)
    for nid in part2_ids:
        G.add_node(nid, bipartite=1)

    for i, e in edges.iterrows():
        src = e["source"]
        tgt = e["target"]
        weight = e["weight"]
        rtype = e["type"]

        if src is None or tgt is None:
            continue

        # Ensure nodes exist in graph (in case relationships reference missing ids)
        if src not in G:
            print("source node not in G")
            G.add_node(src, kind=("event" if src in event_ids else "entity"), bipartite=(0 if src in event_ids else 1))

        if tgt not in G:
            print("target node not in G")
            G.add_node(tgt, kind=("event" if tgt in event_ids else "entity"), bipartite=(0 if tgt in event_ids else 1))

        G.add_edge(src, tgt, weight=weight, type=rtype)

    return G

def connected_components_sorted(G: nx.Graph):
    """Return connected components sorted by size (descending)."""
    comps = list(nx.connected_components(G))
    comps.sort(key=len, reverse=True)
    return comps

In [35]:
G = create_bipartite_graph(events, entities, relationships)
print(f"Loaded graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

from networkx.algorithms import bipartite

#print("Is bipartite ", bipartite.is_bipartite(G))
	# show a few sample edges
#for u, v, d in list(G.edges(data=True))[:1000]:
#	print(u, "-", v, ":", d)

comps = connected_components_sorted(G)

for i, a in attack_scenarios.iterrows():
    c = set(a['event_ids']).union(set(a['entity_ids']))
    print(c in comps)

Loaded graph: 2398 nodes, 3860 edges
True
False
False
False
False
False
